# Phase B Step 6: Demucs Stem Separation (06a, 06b, 06c) — GPU Optimized

Runs Demucs stem separation and residual computation on Kaggle P100 GPU.
Data pulled from HF Hub `Uday-4/djmix-v3`, results pushed back after every mix.

**GPU optimization:** Calls `torch.cuda.empty_cache()` after EVERY segment in 06b (not just between steps).
Reduces OOM risk on long mixes.

**Parallel-safe:** Each notebook writes its own progress file (`phase_b_progress_step6_partN.json`).
Run 2 copies in parallel — one with `MANIFEST = "manifest_100mix_part1"`, the other with `"manifest_100mix_part2"`.

**DRY_RUN:** Set `True` to process only 1 mix (quick error check).

**Runtime:** GPU P100 (Settings → Accelerator → GPU P100). Resumable across 12h sessions.

In [ ]:
!pip install -q --force-reinstall --no-deps demucs
!pip install -q huggingface_hub soundfile librosa cvxpy torchaudio ecos clarabel

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/Uday-461/ai-dj-v4.git', '/kaggle/working/ai-dj/v4'],
    capture_output=True, text=True
)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print('Clone failed:', result.stderr)
else:
    print('Repo ready at /kaggle/working/ai-dj/v4')

!pip install -q -e /kaggle/working/ai-dj/v4
sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os

HF_TOKEN    = "hf_..."                   # <-- paste your HuggingFace token here
HF_REPO     = "Uday-4/djmix-v3"
DATA_ROOT   = "/kaggle/working/djmix"
MANIFEST    = "manifest_100mix_part1"     # <-- Change to "manifest_100mix_part2" for 2nd notebook
DRY_RUN     = False                       # Set True to process only 1 mix (quick error check)

# -------------------------------------------------------
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["AIDJ_DATA_ROOT"] = DATA_ROOT

if HF_TOKEN.startswith("hf_..."):
    raise ValueError("Edit Cell 2: set HF_TOKEN to your real HuggingFace token")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    api.whoami()
    print(f'HF auth OK — repo: {HF_REPO}')
except Exception as e:
    raise RuntimeError(f'HF auth failed: {e}')

In [ ]:
import json
import pickle
import shutil
import tempfile
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

DATA_ROOT_PATH = Path(DATA_ROOT)

# --- Part-scoped progress ---
PROGRESS_KEY = f"phase_b_progress_step6_{MANIFEST.split('_')[-1]}.json"
print(f'Progress file: {PROGRESS_KEY}')


def load_progress():
    """Download this notebook's progress file from HF (or start fresh)."""
    try:
        local = hf_hub_download(
            repo_id=HF_REPO, filename=PROGRESS_KEY,
            repo_type="dataset", token=HF_TOKEN,
            local_dir=DATA_ROOT, force_download=True,
        )
        with open(local) as f:
            p = json.load(f)
        print(f'Loaded progress: {PROGRESS_KEY}')
        return p
    except Exception:
        print('No progress file on HF — starting fresh')
        return {
            "stems_tracks":    [],
            "stems_segments":  [],
            "residuals":       [],
        }


def load_all_step6_progress():
    """Download ALL step6 progress files from HF, merge into union sets."""
    all_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))
    prog_files = [f for f in all_files if f.startswith("phase_b_progress_step6_") and f.endswith(".json")]
    merged = {"stems_tracks": set(), "stems_segments": set(), "residuals": set()}
    for pf in prog_files:
        try:
            local = hf_hub_download(
                repo_id=HF_REPO, filename=pf,
                repo_type="dataset", token=HF_TOKEN,
                local_dir=DATA_ROOT, force_download=True,
            )
            with open(local) as f:
                p = json.load(f)
            for key in merged:
                merged[key].update(p.get(key, []))
        except Exception as e:
            print(f'Warning: could not load {pf}: {e}')
    return merged


def save_progress_local(progress):
    path = DATA_ROOT_PATH / PROGRESS_KEY
    with open(path, 'w') as f:
        json.dump(progress, f, indent=2)
    return path


def push_progress(progress):
    """Upload this notebook's progress file to HF."""
    local_path = save_progress_local(progress)
    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=PROGRESS_KEY,
        repo_id=HF_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
        commit_message=f"Step 6 progress ({MANIFEST.split('_')[-1]})",
    )


# --- Manifest-based mix list ---
def download_manifest_pkls(mix_ids):
    """Download transition PKLs for the given mix IDs from HF."""
    trans_dir = DATA_ROOT_PATH / "results" / "transitions"
    trans_dir.mkdir(parents=True, exist_ok=True)
    for mix_id in mix_ids:
        local = trans_dir / f"{mix_id}.pkl"
        if not local.exists():
            try:
                hf_hub_download(
                    repo_id=HF_REPO, filename=f"results/transitions/{mix_id}.pkl",
                    repo_type="dataset", token=HF_TOKEN,
                    local_dir=DATA_ROOT,
                )
            except Exception as e:
                print(f'Warning: could not download transitions for {mix_id}: {e}')


def build_mix_list_from_manifest():
    """Load manifest JSON from cloned repo, download PKLs, build mix list."""
    manifest_path = f"/kaggle/working/ai-dj/v4/data/{MANIFEST}.json"
    with open(manifest_path) as f:
        manifest = json.load(f)

    mix_ids = [m["id"] for m in manifest["mixes"]]
    print(f'Manifest: {MANIFEST} — {len(mix_ids)} mixes')

    download_manifest_pkls(mix_ids)

    mixes = []
    trans_dir = DATA_ROOT_PATH / "results" / "transitions"
    for m in manifest["mixes"]:
        mix_id = m["id"]
        pkl_path = trans_dir / f"{mix_id}.pkl"
        if not pkl_path.exists():
            print(f'  {mix_id}: no transition PKL, skipping')
            continue
        with open(pkl_path, 'rb') as f:
            transitions = pickle.load(f)
        track_ids = set()
        for t in transitions:
            if t.get("track_id_prev"): track_ids.add(t["track_id_prev"])
            if t.get("track_id_next"): track_ids.add(t["track_id_next"])
        mixes.append({
            "id": mix_id,
            "track_ids": sorted(track_ids),
            "transitions": transitions,
            "n_transitions": len(transitions),
        })
    return mixes


# --- Run ---
progress = load_progress()
all_mixes = build_mix_list_from_manifest()

# Merge all step6 progress to check global completion
merged_progress = load_all_step6_progress()

# Pending = mixes from THIS manifest where any step6 substep is incomplete (in merged view)
done_all = merged_progress["stems_tracks"] & merged_progress["stems_segments"] \
           & merged_progress["residuals"]
pending_mixes = [m for m in all_mixes if m["id"] not in done_all]

if DRY_RUN and pending_mixes:
    pending_mixes = pending_mixes[:1]
    print(f'DRY_RUN: processing only 1 mix')

print(f'\nMixes in manifest: {len(all_mixes)}')
print(f'Globally done (all step6 parts): {len(done_all)}')
print(f'Pending mixes: {len(pending_mixes)}')
for m in pending_mixes:
    steps_done = []
    if m["id"] in merged_progress["stems_tracks"]:   steps_done.append("06a")
    if m["id"] in merged_progress["stems_segments"]: steps_done.append("06b")
    if m["id"] in merged_progress["residuals"]:      steps_done.append("06c")
    print(f'  {m["id"]}: {m["n_transitions"]} transitions, done: {steps_done or ["nothing"]}')

In [ ]:
import sys
if '/kaggle/working/ai-dj/v4' not in sys.path:
    sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import logging
import time
import shutil
import tempfile
import numpy as np
import librosa
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

logging.basicConfig(level=logging.WARNING)

from aidj.stems.separator import StemSeparator
from aidj.stems.stem_cache import StemCache
from aidj.data.residual import compute_residual, align_track_to_mix_segment
from aidj import config


# -------------------------------------------------------
# Helpers
# -------------------------------------------------------

def download_track(tid):
    local = DATA_ROOT_PATH / "tracks" / f"{tid}.mp3"
    if local.exists():
        return local
    local.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id=HF_REPO, filename=f"tracks/{tid}.mp3",
        repo_type="dataset", token=HF_TOKEN,
        local_dir=DATA_ROOT,
    )
    return local


def download_mix_audio(mix_id):
    local = DATA_ROOT_PATH / "mixes" / f"{mix_id}.mp3"
    if local.exists():
        return local
    local.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id=HF_REPO, filename=f"mixes/{mix_id}.mp3",
        repo_type="dataset", token=HF_TOKEN,
        local_dir=DATA_ROOT,
    )
    return local


def upload_mix_outputs(mix, stem_cache):
    """Stage all stems + residuals for this mix into a temp dir and upload to HF."""
    mix_id = mix["id"]
    with tempfile.TemporaryDirectory() as staging_dir:
        staging = Path(staging_dir)
        n_files = 0

        def stage(src, rel):
            nonlocal n_files
            src = Path(src)
            if src.exists():
                dst = staging / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                n_files += 1

        # Track stems
        for tid in mix["track_ids"]:
            stem_dir = stem_cache._stem_dir("tracks", tid)
            if stem_dir.exists():
                for f in stem_dir.iterdir():
                    stage(f, f"stems/tracks/{tid}/{f.name}")

        # Mix segment stems
        for tran in mix["transitions"]:
            tran_id = tran["tran_id"]
            seg_dir = stem_cache._stem_dir("mix_segments", tran_id)
            if seg_dir.exists():
                for f in seg_dir.iterdir():
                    stage(f, f"stems/mix_segments/{tran_id}/{f.name}")

        # Residuals
        res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
        if res_dir.exists():
            for f in res_dir.iterdir():
                stage(f, f"results/residuals/{mix_id}/{f.name}")

        if n_files == 0:
            print(f'  [upload] No files to stage for {mix_id}')
            return

        print(f'  [upload] Uploading {n_files} files for {mix_id}...')
        api.upload_large_folder(
            folder_path=str(staging),
            repo_id=HF_REPO,
            repo_type="dataset",
        )
        print(f'  [upload] Done')


# -------------------------------------------------------
# Step 06a — Separate track stems
# -------------------------------------------------------

def step_06a_track_stems(mix, separator, stem_cache):
    mix_id = mix["id"]
    success = 0
    skipped = 0
    total = len(mix["track_ids"])

    for i, tid in enumerate(mix["track_ids"]):
        if stem_cache.has_stems("tracks", tid):
            skipped += 1
            continue
        try:
            track_path = download_track(tid)
            stem_cache.separate_and_cache(separator, str(track_path), "tracks", tid)
            success += 1
            print(f'  [06a] {mix_id} track {i+1}/{total} ({tid}): OK')
        except Exception as e:
            print(f'  [06a] {mix_id} track {i+1}/{total} ({tid}): FAILED — {e}')

    print(f'  [06a] {mix_id}: {success} separated, {skipped} cached, {total} total')
    return success + skipped, total


# -------------------------------------------------------
# Step 06b — Separate mix segments (GPU OPTIMIZED)
# -------------------------------------------------------

def step_06b_mix_segments(mix, separator, stem_cache):
    """GPU-optimized: calls empty_cache() after EVERY segment, not just between steps."""
    mix_id = mix["id"]
    transitions = mix["transitions"]

    pending = [t for t in transitions
               if not stem_cache.has_stems("mix_segments", t["tran_id"])]
    if not pending:
        print(f'  [06b] {mix_id}: all {len(transitions)} segments already cached')
        return len(transitions), len(transitions)

    mix_path = download_mix_audio(mix_id)
    mix_audio, _ = librosa.load(str(mix_path), sr=config.SR, mono=True)

    success = 0
    skipped = len(transitions) - len(pending)

    for i, tran in enumerate(pending):
        tran_id = tran["tran_id"]
        mix_cue_in  = tran.get("mix_cue_in_time_next")
        mix_cue_out = tran.get("mix_cue_out_time_prev")
        if mix_cue_in is None or mix_cue_out is None:
            print(f'  [06b] {tran_id}: missing cue times, skip')
            continue

        mix_start = int(min(mix_cue_in, mix_cue_out) * config.SR)
        mix_end   = int(max(mix_cue_in, mix_cue_out) * config.SR)
        if mix_end - mix_start < config.SR:
            print(f'  [06b] {tran_id}: segment too short, skip')
            continue

        segment = mix_audio[mix_start:mix_end]
        try:
            stem_cache.separate_and_cache_segment(
                separator, segment, "mix_segments", tran_id
            )
            success += 1
            if success % 10 == 0:
                print(f'  [06b] {mix_id}: {success}/{len(pending)} segments done')
        except Exception as e:
            print(f'  [06b] {tran_id}: FAILED — {e}')
        finally:
            # GPU OPTIMIZATION: clear cache after EVERY segment
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f'  [06b] {mix_id}: {success} new, {skipped} cached, {len(transitions)} total')
    return success + skipped, len(transitions)


# -------------------------------------------------------
# Step 06c — Compute residuals
# -------------------------------------------------------

def step_06c_residuals(mix, stem_cache):
    mix_id = mix["id"]
    transitions = mix["transitions"]
    res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
    res_dir.mkdir(parents=True, exist_ok=True)

    success = 0
    skipped = 0

    for tran in transitions:
        tran_id  = tran["tran_id"]
        prev_id  = tran["track_id_prev"]
        next_id  = tran["track_id_next"]
        out_path = res_dir / f"{tran_id}.npz"

        if out_path.exists():
            skipped += 1
            continue

        mix_cue_in  = tran.get("mix_cue_in_time_next")
        mix_cue_out = tran.get("mix_cue_out_time_prev")
        if mix_cue_in is None or mix_cue_out is None:
            continue

        region_len = int((max(mix_cue_in, mix_cue_out) - min(mix_cue_in, mix_cue_out)) * config.SR)
        if region_len < config.SR:
            continue

        mix_seg_stems    = stem_cache.load_stems("mix_segments", tran_id)
        prev_track_stems = stem_cache.load_stems("tracks", prev_id)
        next_track_stems = stem_cache.load_stems("tracks", next_id)

        if mix_seg_stems is None:
            print(f'  [06c] {tran_id}: missing mix segment stems, skip')
            continue

        residual_data = {}
        if prev_track_stems is not None:
            t_start = tran.get("track_cue_in_time_prev", 0)
            aligned = {s: align_track_to_mix_segment(
                prev_track_stems[s], t_start, region_len, config.SR
            ) for s in config.STEMS if s in prev_track_stems}
            for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                residual_data[f"{stem}_prev"] = spec

        if next_track_stems is not None:
            t_start = tran.get("track_cue_in_time_next", 0)
            aligned = {s: align_track_to_mix_segment(
                next_track_stems[s], t_start, region_len, config.SR
            ) for s in config.STEMS if s in next_track_stems}
            for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                residual_data[f"{stem}_next"] = spec

        if residual_data:
            np.savez_compressed(str(out_path), **residual_data)
            success += 1

    print(f'  [06c] {mix_id}: {success} computed, {skipped} cached, {len(transitions)} total')
    return success + skipped, len(transitions)


# -------------------------------------------------------
# Main processing loop
# -------------------------------------------------------

separator  = StemSeparator(device="cuda" if torch.cuda.is_available() else "cpu")
stem_cache = StemCache(DATA_ROOT_PATH)
print(f'Separator device: {separator.device}')
print(f'Mixes to process: {len(pending_mixes)}')

session_start = time.time()

for mix_idx, mix in enumerate(pending_mixes):
    mix_id = mix["id"]
    mix_start_t = time.time()
    print(f'\n{"="*60}')
    print(f'[{mix_idx+1}/{len(pending_mixes)}] {mix_id} ({len(mix["track_ids"])} tracks, {mix["n_transitions"]} transitions)')
    print(f'{"="*60}')

    # Step 06a: track stems (GPU)
    if mix_id not in progress["stems_tracks"]:
        print(f'\n--- Step 06a: track stems ---')
        step_06a_track_stems(mix, separator, stem_cache)
        progress["stems_tracks"].append(mix_id)
        push_progress(progress)
    else:
        print(f'[06a] {mix_id}: already done, skipping')

    # Flush GPU cache between 06a and 06b to avoid OOM
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Step 06b: mix segment stems (GPU) — GPU OPTIMIZED
    if mix_id not in progress["stems_segments"]:
        print(f'\n--- Step 06b: mix segment stems ---')
        step_06b_mix_segments(mix, separator, stem_cache)  # clears cache after each segment
        progress["stems_segments"].append(mix_id)
        push_progress(progress)
    else:
        print(f'[06b] {mix_id}: already done, skipping')

    # Step 06c: residuals (CPU)
    if mix_id not in progress["residuals"]:
        print(f'\n--- Step 06c: residuals ---')
        step_06c_residuals(mix, stem_cache)
        progress["residuals"].append(mix_id)
        push_progress(progress)
    else:
        print(f'[06c] {mix_id}: already done, skipping')

    # Upload stems + residuals to HF
    print(f'\n--- Upload for {mix_id} ---')
    upload_mix_outputs(mix, stem_cache)

    elapsed = (time.time() - mix_start_t) / 60
    total_elapsed = (time.time() - session_start) / 3600
    mixes_left = len(pending_mixes) - (mix_idx + 1)
    avg_per_mix = (time.time() - session_start) / (mix_idx + 1) / 60
    eta_min = mixes_left * avg_per_mix
    print(f'\n[{mix_idx+1}/{len(pending_mixes)}] {mix_id} done in {elapsed:.1f} min | '
          f'session: {total_elapsed:.2f}h | ETA: {eta_min:.0f} min ({mixes_left} left)')

print(f'\n{"="*60}')
print(f'Step 6 complete for {MANIFEST}!')
print(f'  stems_tracks:   {len(progress["stems_tracks"])} mixes')
print(f'  stems_segments: {len(progress["stems_segments"])} mixes')
print(f'  residuals:      {len(progress["residuals"])} mixes')

In [ ]:
# Load all step6 progress files and show combined view
merged = load_all_step6_progress()

print(f'Combined Step 6 progress (all parts):')
print(f'  stems_tracks:   {len(merged["stems_tracks"])} mixes')
print(f'  stems_segments: {len(merged["stems_segments"])} mixes')
print(f'  residuals:      {len(merged["residuals"])} mixes')

fully_done = merged["stems_tracks"] & merged["stems_segments"] & merged["residuals"]
print(f'  Fully complete:  {len(fully_done)} mixes')

# Count files on HF
from collections import defaultdict
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))
stems_tracks   = [f for f in all_hf_files if f.startswith("stems/tracks/")      and f.endswith(".ogg")]
stems_segments = [f for f in all_hf_files if f.startswith("stems/mix_segments/") and f.endswith(".ogg")]
residuals      = [f for f in all_hf_files if f.startswith("results/residuals/")  and f.endswith(".npz")]

print(f'\nHF Hub file counts:')
print(f'  stems/tracks/:       {len(stems_tracks)} OGG files')
print(f'  stems/mix_segments/: {len(stems_segments)} OGG files')
print(f'  results/residuals/:  {len(residuals)} NPZ files')

# Show which mixes from THIS manifest are done vs pending
this_manifest_ids = {m["id"] for m in all_mixes}
done_in_manifest = fully_done & this_manifest_ids
print(f'\nThis manifest ({MANIFEST}): {len(done_in_manifest)}/{len(this_manifest_ids)} fully done')
remaining = this_manifest_ids - fully_done
if remaining:
    print(f'  Remaining: {sorted(remaining)[:20]}{"..." if len(remaining) > 20 else ""}')